In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, r2_score

In [7]:
df = pd.read_csv('data.csv')
df.head()

,Location,Commodity,Price,Temperature,Rainfall,Humidity,Unnamed: 6
0,Lagos,Rice,4000,30,5,70,NaN
1,Abuja,Tomatoes,250,28,3,65,NaN
2,Kano,Beans,3000,32,2,60,NaN
3,Port Harcourt,Yam,1500,29,4,68,NaN
4,Ibadan,Plantain,1000,31,6,72,NaN


In [8]:
print(df.shape)
print(df.info())
print(df.isnull().sum())
print(df.duplicated().sum())
df.describe()

(44, 7)
<class 'pandas.DataFrame'>
RangeIndex: 44 entries, 0 to 43
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Location     44 non-null     str    
 1   Commodity    44 non-null     str    
 2   Price        44 non-null     int64  
 3   Temperature  44 non-null     int64  
 4   Rainfall     44 non-null     int64  
 5   Humidity     44 non-null     int64  
 6   Unnamed: 6   0 non-null      float64
dtypes: float64(1), int64(4), str(2)
memory usage: 3.1 KB
None
Location        0
Commodity       0
Price           0
Temperature     0
Rainfall        0
Humidity        0
Unnamed: 6     44
dtype: int64
0


,Price,Temperature,Rainfall,Humidity,Unnamed: 6
count,44.000000,44.000000,44.000000,44.000000,0.0
mean,1547.727273,29.500000,3.522727,68.295455,NaN
std,975.451866,1.745426,1.704831,4.830039,NaN
min,250.000000,27.000000,1.000000,60.000000,NaN
25%,800.000000,28.000000,2.000000,65.000000,NaN
50%,1350.000000,29.500000,3.500000,69.000000,NaN
75%,2125.000000,31.000000,5.000000,72.000000,NaN
max,4000.000000,33.000000,6.000000,75.000000,NaN


In [11]:
df.drop_duplicates(inplace=True)

num_cols = df.select_dtypes(include=['int64','float64']).columns
cat_cols = df.select_dtypes(include=['object']).columns

df[num_cols] = df[num_cols].fillna(df[num_cols].median())
df[cat_cols] = df[cat_cols].fillna(df[cat_cols].mode().iloc[0])

le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col])

df.head()

,Location,Commodity,Price,Temperature,Rainfall,Humidity,Unnamed: 6
0,28,15,4000,30,5,70,NaN
1,2,17,250,28,3,65,NaN
2,24,0,3000,32,2,60,NaN
3,38,18,1500,29,4,68,NaN
4,16,14,1000,31,6,72,NaN


In [14]:
df.drop(columns=[col for col in df.columns if 'Unnamed' in col], inplace=True)

X = df.drop('Price', axis=1)
y = df['Price']

# Use 70/30 split for small dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train:", X_train.shape, "| Test:", X_test.shape)

Train: (30, 5) | Test: (14, 5)


In [16]:
models = {
    'Linear Regression': LinearRegression(),
    'KNN': KNeighborsRegressor(n_neighbors=3),
    'Decision Tree': DecisionTreeRegressor(max_depth=3, random_state=42),
    'SVR': SVR(C=100, epsilon=50)
}

result = []

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    result.append({
        'Model': name,
        'MAE': round(mean_absolute_error(y_test, y_pred), 4)
    })

result

[{'Model': 'Linear Regression', 'MAE': 616.0765},
 {'Model': 'KNN', 'MAE': 672.619},
 {'Model': 'Decision Tree', 'MAE': 1025.5952},
 {'Model': 'SVR', 'MAE': 581.5137}]

In [18]:
df

,Location,Commodity,Price,Temperature,Rainfall,Humidity
0,28,15,4000,30,5,70
1,2,17,250,28,3,65
2,24,0,3000,32,2,60
3,38,18,1500,29,4,68
4,16,14,1000,31,6,72
5,23,11,2000,27,1,75
6,11,1,800,30,5,70
7,9,12,300,28,3,65
8,39,8,1200,33,2,60
9,22,7,1800,29,4,68


In [17]:
import joblib

joblib.dump(models['SVR'], 'food_price_model.pkl')
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(X.columns.to_list(), 'columns.pkl')

print('✅ Model saved!')
print('✅ Scaler saved!')
print('✅ Columns saved!')

✅ Model saved!
✅ Scaler saved!
✅ Columns saved!
